# HDF5 local processing

Read, process, and visualise frames from local HDF5 files using `LocalBackend` and `HDF5Reader`.
The backend handles file discovery; the reader handles HDF5 parsing. They are decoupled and
interact only through the `open_fileobj` interface.

In [ ]:
from pathlib import Path

import numpy as np

from pd_xray.data import HDF5Reader, LocalBackend
from pd_xray.processing import ImageProcessor
from pd_xray.visualisation import view_frames, apply_and_view, save_selection

## Configuration

In [ ]:
DATA_DIR     = Path("../data")
HDF5_FILE    = "scan.h5"
DATASET_PATH = None     # set to e.g. "/entry/data/data" if auto-detect fails
OUTPUT_DIR   = Path("../output")

FRAME_START = 0
FRAME_END   = 20

## Inspect

In [ ]:
backend = LocalBackend(DATA_DIR)
backend.connect()

for f in backend.list_files(pattern="*.h5"):
    print(f"{f.path}  ({f.size_bytes / 1024**3:.2f} GB)")

In [ ]:
reader = HDF5Reader()
info = reader.inspect(DATA_DIR / HDF5_FILE)

for g in info["groups"]:
    print(f"  [group]   {g}")
for ds_path, meta in info["datasets"].items():
    shape_str = " x ".join(str(s) for s in meta["shape"])
    print(f"  [dataset] {ds_path}   {shape_str}   {meta['dtype']}")

In [ ]:
header = reader.read_header(DATA_DIR / HDF5_FILE, dataset_path=DATASET_PATH)
print(f"dataset : {header['dataset_path']}")
print(f"shape   : {header['shape']}")
print(f"dtype   : {header['dtype']}")
print(f"frames  : {header['n_frames']}")

## Load and view

For local files, `lazy_open(path)` gives h5py native C-level file I/O (the sec2 driver).
`lazy_open_remote(backend, file)` routes every internal chunk read through a Python file
object, which can be 10-30x slower on files with many small chunks. Use `lazy_open_remote`
only when the file is not accessible by path (S3, SFTP).

In [ ]:
with reader.lazy_open(backend.root / HDF5_FILE, dataset_path=DATASET_PATH) as arr:
    print(arr)
    frame = arr[0]

view_frames(frame, title="frame 0")

In [ ]:
with reader.lazy_open(backend.root / HDF5_FILE, dataset_path=DATASET_PATH) as arr:
    frames = arr[FRAME_START:FRAME_END]

view_frames(frames, title=f"frames {FRAME_START}:{FRAME_END}")

`lazy_read` is an alternative when you want to open the file once and access it across multiple
cells. Used as a context manager it keeps the h5py file handle open for the block, equivalent
to `lazy_open`.

In [ ]:
arr = reader.lazy_read(DATA_DIR / HDF5_FILE, dataset_path=DATASET_PATH)

with arr:
    frame  = arr[0]
    frames = arr[FRAME_START:FRAME_END]

## Process

In [ ]:
proc = (
    ImageProcessor()
    .normalise()
    .gaussian_blur(sigma=1.0)
)

processed = apply_and_view(frames, proc, title="processed")

### Available ImageProcessor steps

All steps are chainable via the fluent builder. 3D stacks are handled automatically;
operations that are inherently 2D (bilateral, rolling ball, CLAHE, rotate) are applied
per slice. Build your pipeline by mixing and matching from the table below.

**Intensity**

| Step | Key parameters | Notes |
|---|---|---|
| `.normalise()` | `low=None, high=None` | rescales to [0, 1] by default |
| `.clip()` | `vmin, vmax` | hard clamp |
| `.rescale_intensity()` | `p1=2.0, p99=98.0` | percentile-based contrast stretch |
| `.clahe()` | `clip_limit=3.0, tile_grid_size=(8,8)` | adaptive histogram equalisation |
| `.log_transform()` | `epsilon=1e-8` | negative log, suits absorption contrast |
| `.local_contrast_enhancement()` | `sigma=1.0` | divides by local mean/std |

**Spatial filters**

| Step | Key parameters | Notes |
|---|---|---|
| `.gaussian_blur()` | `sigma` | Gaussian smoothing |
| `.median_filter()` | `size` | impulse noise removal |
| `.bilateral_filter()` | `sigma_spatial, sigma_color` | edge-preserving smoothing |
| `.rolling_ball()` | `radius=50, smoothing_sigma=2.0` | background subtraction |

**Morphological (binary inputs)**

| Step | Key parameters | Notes |
|---|---|---|
| `.erode()` | `kernel_size, iterations=1` | |
| `.dilate()` | `kernel_size, iterations=1` | |
| `.open()` | `kernel_size, iterations=1` | erode then dilate |
| `.close()` | `kernel_size, iterations=1` | dilate then erode |
| `.fill_holes()` | | fill enclosed background regions |
| `.remove_small_objects()` | `min_size=64` | |
| `.remove_small_holes()` | `area_threshold=64` | |

**Geometric**

| Step | Key parameters | Notes |
|---|---|---|
| `.crop()` | `row_start, row_end, col_start, col_end` | `slice_start/end` for Z on 3D |
| `.resize()` | `height, width, depth=None` | bilinear interpolation |
| `.rotate()` | `angle_deg` | counter-clockwise, about centre |
| `.circular_mask()` | `mask_ratio=0.5, crop_to_circle=False` | circular FOV mask |
| `.cylindrical_mask()` | `mask_ratio=0.5` | mask + crop to bounding box |

Every step accepts an optional `dtype` keyword to cast the output of that step:
```python
proc = (
    ImageProcessor()
    .normalise(dtype=np.float32)
    .gaussian_blur(sigma=1.5)
    .crop(row_start=256, row_end=1800, col_start=512, col_end=3584)
)
```

## Save

### Save as Numpy array

You can save your data as numpy array, which can be useful for loading and processing later on.

In [ ]:
save_selection(processed, OUTPUT_DIR / "processed_frames")

### Save as TIFF

Alternatively, you can save the frames as TIFF images, which might be easier to visualise in some cases.

NOTE that this works for a single frame, if you want to save multiple frames, you need to loop them.

In [ ]:
from pd_xray.data import TIFFReader

tiff_reader = TIFFReader()

frame_number = 0  # Which frame you want to save

tiff_reader.write_as(
    path = OUTPUT_DIR / f"processed_frame_{frame_number}.tiff",
    data=processed[frame_number],
    fmt="tiff"
)

### Save as jpeg

Similar to TIFF, frames can be saved as JPEG

In [ ]:
tiff_reader.write_as(
    path = OUTPUT_DIR / f"processed_frame_{frame_number}.jpeg",
    data=processed[frame_number],
    fmt="jpeg"
)

In [ ]:
backend.disconnect()